In [0]:
%pip install openai
dbutils.library.restartPython()

In [0]:
from openai import OpenAI

client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url=f"{BASE_URL}/mlflow/v1"
)

In [0]:
import pyspark.sql.functions as F

In [0]:
df = (
    spark.read.option("header", "true")
    .format("csv")
    .load("/Volumes/ai_lab_demo/source/source_volume/B1_1_German_Vocabulary_German_English_Urdu.csv")
)

In [0]:
df_german = (
    df.withColumn(
        "query",
        F.concat(
            F.lit(
                "Erstelle einen natürlichen deutschen Beispielsatz "
                "auf B1-Niveau mit diesem Wort: "
            ),
            F.col("German"),
            F.lit(
                ". Verwende das Wort genau im Satz. "
                "Gib nur einen Satz aus, ohne Erklärung und ohne Übersetzung."
            )
        )
    )
)

In [0]:
df_german = (
    df_german.withColumn(
        "B1_german_example_sentence",
        F.expr(
            f"""
            ai_query(
                '{MODEL}',
                query
            )
            """
        )
    )
)

In [0]:
df_german = (
    df_german.withColumn(
        "B1_german_example_sentence",
        F.expr(
            """
            ai_query(
                'databricks-meta-llama-3-3-70b-instruct',
                query
            )
            """
        )
    )
)

In [0]:
df_german.display()

In [0]:
df_german = df_german.select("German", "B1_german_example_sentence", "English", "Urdu")

In [0]:
df_german.write.mode("overwrite").option("header", "true").format("csv").save("/Volumes/ai_lab_demo/source/source_volume/b1_translate")